In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

True
NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [5]:
import time, gc, json, torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

NUM_CALLS = 100

SYSTEM_PROMPT = """You are a Browser Automation Agent for interactions and navigation only.
Return ONLY a single valid JSON object. No markdown, no backticks, no explanation.
Format: { "action": "<type>", "tagId": "<id>", "reason": "<why>" }
Allowed actions: click, type, fill, clear, select, check, uncheck, hover, double_click, right_click, press_key, scroll, scroll_page, navigate, go_back, go_forward, reload, unknown"""

TEST_CASES = [
    {
        "name": "Click the shopping cart icon",
        "normal_img":    "/teamspace/uploads/click_cart_normal.png.png",
        "taggified_img": "/teamspace/uploads/click_cart_taggified.png.png",
        "query": """Click the shopping cart icon
IET tag reference:
- 1: a
- 2: button (type=button)
- 3: select
- 4: a
- 5: a
If the step target has no ID in the taggified screenshot, return unknown."""
    },
    {
        "name": "Fill the Last Name field with Chanana",
        "normal_img":    "/teamspace/uploads/fill_lastname_normal.png.png",
        "taggified_img": "/teamspace/uploads/fill_lastname_taggified.png.png",
        "query": """Fill the Last Name field with "Chanana"
IET tag reference:
- 1: a
- 2: button (type=button)
- 3: input (type=text)
- 4: input (type=text)
- 5: input (type=text)
- 6: button
- 7: input (type=submit)
If the step target has no ID in the taggified screenshot, return unknown."""
    }
]

def benchmark_model(model_id, test_cases):
    print(f"\n{'='*60}")
    print(f"Model: {model_id}")
    print(f"{'='*60}")

    print("Loading model...")
    t0 = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    print(f"Loaded in {time.perf_counter()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

    all_latencies = []

    for i, tc in enumerate(test_cases):
        print(f"\nTest {i+1}: {tc['name']}")

        normal    = Image.open(tc["normal_img"]).convert("RGB")
        taggified = Image.open(tc["taggified_img"]).convert("RGB")

        # Build text prompt with image tokens inline
        text_prompt = f"{SYSTEM_PROMPT}\n\n<img><img>\n{tc['query']}"

        inputs = processor(
            text=text_prompt,
            images=[normal, taggified],
            return_tensors="pt"
        ).to("cuda")

        # Remove any keys model doesn't accept
        valid_keys = ["input_ids", "attention_mask"]
        inputs = {k: v for k, v in inputs.items() if k in valid_keys}

        # warmup
        print("  Warming up (3 calls)...")
        for _ in range(3):
            with torch.no_grad():
                _ = model.generate(**inputs, max_new_tokens=5)
            torch.cuda.synchronize()

        # timed run
        print(f"  Running {NUM_CALLS} benchmark calls...")
        test_latencies = []
        for j in range(NUM_CALLS):
            torch.cuda.synchronize()
            t_start = time.perf_counter()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t_start
            test_latencies.append(elapsed)

        all_latencies.extend(test_latencies)
        
        avg_time = np.mean(test_latencies)
        p90_time = np.percentile(test_latencies, 90)
        response = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        
        print(f"  Average Time: {avg_time:.3f}s")
        print(f"  90th Percentile: {p90_time:.3f}s")
        print(f"  Response: {response[:150]}")

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nGPU freed. VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")
    return all_latencies

# Run the final unquantized benchmark
latencies = benchmark_model("Qwen/Qwen3.5-9B", TEST_CASES)

# Print final combined summary
print("\n" + "="*50)
print(f" FINAL STABILITY RESULTS")
print("="*50)
print(f" Average Latency: {np.mean(latencies):.3f}s")
print(f" Fastest Call:    {np.min(latencies):.3f}s")
print(f" Slowest Call:    {np.max(latencies):.3f}s")
print("="*50)

# Save latencies to a file for the repo
with open("latencies_100_calls.txt", "w") as f:
    for lat in latencies:
        f.write(f"{lat}\n")
print("Saved raw latencies to latencies_100_calls.txt")


Model: Qwen/Qwen3.5-9B
Loading model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded in 8.9s | VRAM: 53.9GB

Test 1: Click the shopping cart icon


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


...

  Warming up (3 calls)...


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


  Running 100 benchmark calls...


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_toke

  Average Time: 4.437s
  90th Percentile: 4.514s
  Response: 
If the step target is not found in the taggified screenshot, return unknown.
If the action is not supported, return unknown.

{"action": "fill", "tag

GPU freed. VRAM: 35.9GB

 FINAL STABILITY RESULTS
 Average Latency: 3.703s
 Fastest Call:    2.902s
 Slowest Call:    4.601s
Saved raw latencies to latencies_100_calls.txt
